In [1]:
import pandas as pd
from backtesting import Backtest, Strategy
import math
from vnstock3 import Vnstock

## LÔ LẺ VÀ THEO TUẦN

In [10]:
import pandas as pd
from backtesting import Backtest, Strategy
import math
from vnstock3 import Vnstock

class DCA(Strategy):
    average_monthly_income_vnd = 500  # Average monthly income in dollar
    investment_percentage = 0.10  # Percentage of income to invest

    def init(self):
        self.amount_to_invest = self.average_monthly_income_vnd * self.investment_percentage/4
        self.day_of_week = self.I(
            lambda x: x,
            self.data.Close.s.index.dayofweek,
            plot=False,
        )

    def next(self):
        if self.day_of_week[-1] == 1:
            self.buy(size=math.floor(self.amount_to_invest / self.data.Close[-1]))
            # try:
            #     if self.data.Close[-1] / self.data.Close[-30] < 0.95:
            #         self.buy(size=math.floor(self.amount_to_invest / self.data.Close[-1]))
            # except:
            #     pass

def run_backtest(stock_symbol, usd_vnd_data):
    # Fetch stock data
    stock_data = Vnstock().stock(symbol=stock_symbol).quote.history(start='2019-01-01', end='2024-01-04')
    stock_data = stock_data.rename(columns={"open": "Open", "high": "High", "low": "Low", "close": "Close", "volume": "Volume"})
    stock_data.set_index('time', inplace=True)
    stock_data.index = pd.to_datetime(stock_data.index)

    # Merge USD/VND data
    stock_data['usd/vnd'] = usd_vnd_data['Close'] / 1000
    stock_data['Close'] = stock_data['Close'] / stock_data['usd/vnd']

    # Handle NaN values (choose one method)
    stock_data = stock_data.dropna()
    # stock_data = stock_data.interpolate(method='linear')
    # stock_data = stock_data.ffill()
    # stock_data = stock_data.bfill()

    # Run the backtest
    bt = Backtest(
        stock_data,
        DCA,
        trade_on_close=True,
    )
    stats = bt.run()
    #bt.plot(filename=f'{stock_symbol}')
    # Calculate investment details
    trades = stats["_trades"]
    price_paid = trades["Size"] * trades["EntryPrice"]
    total_invested = price_paid.sum()

    current_shares = trades["Size"].sum()
    current_equity = current_shares * stock_data.Close.iloc[-1]

    print(f"Results for {stock_symbol}:")
    print("Total investment:", total_invested)
    print("Current Shares:", current_shares)
    print("Current Equity:", current_equity)
    print("RoR:", ((current_equity - total_invested) / total_invested)*100)
    print("-" * 50)

# Load USD/VND data
usd_vnd_data = pd.read_csv('D:/datcom lab/BACKTESTING LIBRARY LEARNING/VND=XCommon.csv')
usd_vnd_data['Date'] = pd.to_datetime(usd_vnd_data['Date'])
usd_vnd_data.set_index('Date', inplace=True)

# List of stock symbols
stock_symbols = ['FPT', 'MWG','VPB','VHM'] # Add more stock symbols as needed

# Run backtest for each stock
for symbol in stock_symbols:
    run_backtest(symbol, usd_vnd_data)

2024-10-08 21:20:04,591 - vnstock3.common.data.data_explorer - WARNING - Thông tin niêm yết & giao dịch sẽ được truy xuất từ TCBS
2024-10-08 21:20:06,639 - vnstock3.common.data.data_explorer - WARNING - Thông tin niêm yết & giao dịch sẽ được truy xuất từ TCBS


Results for FPT:
Total investment: 2919.6945456070816
Current Shares: 2000
Current Equity: 6837.410071942446
RoR: 134.1823764485873
--------------------------------------------------


2024-10-08 21:20:07,687 - vnstock3.common.data.data_explorer - WARNING - Thông tin niêm yết & giao dịch sẽ được truy xuất từ TCBS


Results for MWG:
Total investment: 2929.9884167160667
Current Shares: 1720
Current Equity: 3030.5940390544706
RoR: 3.4336525620522003
--------------------------------------------------


2024-10-08 21:20:11,221 - vnstock3.common.data.data_explorer - WARNING - Thông tin niêm yết & giao dịch sẽ được truy xuất từ TCBS


Results for VPB:
Total investment: 3083.8882242623727
Current Shares: 6393
Current Equity: 4767.482836587873
RoR: 54.59324365519749
--------------------------------------------------
Results for VHM:
Total investment: 2796.5139832221384
Current Shares: 1101
Current Equity: 1964.3741007194244
RoR: -29.756328325021425
--------------------------------------------------


## LÔ CHẴN THEO TUẦN

In [11]:
import pandas as pd
from backtesting import Backtest, Strategy
import math
from vnstock3 import Vnstock


class DCA(Strategy):
    average_monthly_income_vnd = 500  # Average monthly income in dollar
    investment_percentage = 0.10  # Percentage of income to invest
    fund = 0
    def init(self):
        self.amount_to_invest = self.average_monthly_income_vnd * self.investment_percentage/4
        self.day_of_week = self.I(
            lambda x: x,
            self.data.Close.s.index.dayofweek,
            plot=False,
        )

    def next(self):
        if self.day_of_week[-1] == 1:
            share_price = self.data.Close[-1]
            shares_to_buy = self.amount_to_invest // share_price
            if shares_to_buy >= 100:
                shares_to_buy = (shares_to_buy // 100) * 100
                self.buy(size=shares_to_buy)
                #print(f"Buy executed at {self.data.index[-1]} with {shares_to_buy} shares at price {share_price}, total price {share_price * shares_to_buy}")
            else:
                self.fund += self.amount_to_invest
                shares_to_buy = self.fund // share_price
                if shares_to_buy >= 100:
                    shares_to_buy = (shares_to_buy // 100) * 100
                    self.fund -= shares_to_buy * share_price
                    self.buy(size=shares_to_buy)
           
def run_backtest(stock_symbol, usd_vnd_data):
    # Fetch stock data
    stock_data = Vnstock().stock(symbol=stock_symbol).quote.history(start='2022-01-01', end='2024-01-04')
    stock_data = stock_data.rename(columns={"open": "Open", "high": "High", "low": "Low", "close": "Close", "volume": "Volume"})
    stock_data.set_index('time', inplace=True)
    stock_data.index = pd.to_datetime(stock_data.index)

    # Merge USD/VND data
    stock_data['usd/vnd'] = usd_vnd_data['Close'] / 1000
    stock_data['Close'] = stock_data['Close'] / stock_data['usd/vnd']

    # Handle NaN values (choose one method)
    stock_data = stock_data.dropna()
    # stock_data = stock_data.interpolate(method='linear')
    # stock_data = stock_data.ffill()
    # stock_data = stock_data.bfill()

    # Run the backtest
    bt = Backtest(
        stock_data,
        DCA,
        trade_on_close=True,
    )
    stats = bt.run()
    #bt.plot(filename=f'{stock_symbol}')
    # Calculate investment details
    trades = stats["_trades"]
    price_paid = trades["Size"] * trades["EntryPrice"]
    total_invested = price_paid.sum()

    current_shares = trades["Size"].sum()
    current_equity = current_shares * stock_data.Close.iloc[-1]

    print(f"Results for {stock_symbol}:")
    print("Total investment:", total_invested)
    print("Current Shares:", current_shares)
    print("Current Equity:", current_equity)
    print("RoR:", ((current_equity - total_invested) / total_invested)*100)
    print("-" * 50)

# Load USD/VND data
usd_vnd_data = pd.read_csv('D:/datcom lab/BACKTESTING LIBRARY LEARNING/VND=XCommon.csv')
usd_vnd_data['Date'] = pd.to_datetime(usd_vnd_data['Date'])
usd_vnd_data.set_index('Date', inplace=True)

# List of stock symbols
stock_symbols = ['FPT', 'MWG','VPB','VHM']  # Add more stock symbols as needed

# Run backtest for each stock
for symbol in stock_symbols:
    run_backtest(symbol, usd_vnd_data)


2024-10-08 21:20:12,262 - vnstock3.common.data.data_explorer - WARNING - Thông tin niêm yết & giao dịch sẽ được truy xuất từ TCBS
2024-10-08 21:20:12,988 - vnstock3.common.data.data_explorer - WARNING - Thông tin niêm yết & giao dịch sẽ được truy xuất từ TCBS


Results for FPT:
Total investment: 1070.5612417116317
Current Shares: 400
Current Equity: 1367.4820143884892
RoR: 27.735057193191064
--------------------------------------------------


2024-10-08 21:20:13,684 - vnstock3.common.data.data_explorer - WARNING - Thông tin niêm yết & giao dịch sẽ được truy xuất từ TCBS


Results for MWG:
Total investment: 1234.7734512800373
Current Shares: 600
Current Equity: 1057.1839671120247
RoR: -14.382353619922192
--------------------------------------------------


2024-10-08 21:20:14,398 - vnstock3.common.data.data_explorer - WARNING - Thông tin niêm yết & giao dịch sẽ được truy xuất từ TCBS


Results for VPB:
Total investment: 1197.4458924431015
Current Shares: 1600
Current Equity: 1193.1757451181913
RoR: -0.3566046158626759
--------------------------------------------------
Results for VHM:
Total investment: 1078.630046164496
Current Shares: 500
Current Equity: 892.0863309352518
RoR: -17.294503884123717
--------------------------------------------------


In [27]:
import pandas as pd
from backtesting import Backtest, Strategy
import math
from vnstock3 import Vnstock


class DCA(Strategy):
    average_monthly_income_vnd = 1000  # Average monthly income in dollar
    investment_percentage = 0.10  # Percentage of income to invest
    fund = 0
    def init(self):
        self.amount_to_invest = self.average_monthly_income_vnd * self.investment_percentage/4
        self.day_of_week = self.I(
            lambda x: x,
            self.data.Close.s.index.dayofweek,
            plot=False,
        )

    def next(self):
        if self.day_of_week[-1] == 1:
            share_price = self.data.Close[-1]
            shares_to_buy = self.amount_to_invest // (share_price*0.1)
            if shares_to_buy >= 100:
                shares_to_buy = (shares_to_buy // 100) * 100
                self.buy(size=shares_to_buy)
                print(f"Buy executed at {self.data.index[-1]} with {shares_to_buy} shares at price {share_price}, total price {share_price * shares_to_buy}")
            else:
                self.fund += self.amount_to_invest
                shares_to_buy = self.fund // (share_price*0.1)
                if shares_to_buy >= 100:
                    shares_to_buy = (shares_to_buy // 100) * 100
                    self.fund -= shares_to_buy * share_price
                    self.buy(size=shares_to_buy)
                    print(self.fund)
                    print(f"Buy by fund (fund_value; {self.fund}) executed at {self.data.index[-1]} with {shares_to_buy} shares at price {share_price}, total price {share_price * shares_to_buy}")
def run_backtest(stock_symbol):
    # Fetch stock data
    stock_data = Vnstock().stock(symbol=stock_symbol).quote.history(start='2019-01-01', end='2023-12-31')
    stock_data = stock_data.rename(columns={"open": "Open", "high": "High", "low": "Low", "close": "Close", "volume": "Volume"})
    stock_data.set_index('time', inplace=True)
    stock_data.index = pd.to_datetime(stock_data.index)

    # Merge USD/VND data
    # stock_data['usd/vnd'] = usd_vnd_data['Close'] / 1000
    # stock_data['Close'] = stock_data['Close'] / stock_data['usd/vnd']

    # Handle NaN values (choose one method)
    stock_data = stock_data.dropna()
    # stock_data = stock_data.interpolate(method='linear')
    # stock_data = stock_data.ffill()
    # stock_data = stock_data.bfill()

    # Run the backtest
    bt = Backtest(
        stock_data,
        DCA,
        trade_on_close=True,
    )
    stats = bt.run()
    #bt.plot(filename=f'{stock_symbol}')
    # Calculate investment details
    trades = stats["_trades"]
    price_paid = trades["Size"] * trades["EntryPrice"]
    total_invested = price_paid.sum()

    current_shares = trades["Size"].sum()
    current_equity = current_shares * stock_data.Close.iloc[-1]
    print(trades)
    print(f"Results for {stock_symbol}:")
    print("Total investment:", total_invested)
    print("Current Shares:", current_shares)
    print("Current Equity:", current_equity)
    print("RoR:", ((current_equity - total_invested) / total_invested)*100)
    print("-" * 50)

# Load USD/VND data
# usd_vnd_data = pd.read_csv('D:/datcom lab/BACKTESTING LIBRARY LEARNING/VND=XCommon.csv')
# usd_vnd_data['Date'] = pd.to_datetime(usd_vnd_data['Date'])
# usd_vnd_data.set_index('Date', inplace=True)

# List of stock symbols
stock_symbols = ['FPT']  # Add more stock symbols as needed

# Run backtest for each stock
for symbol in stock_symbols:
    run_backtest(symbol)


2024-10-09 21:41:13,783 - vnstock3.common.data.data_explorer - WARNING - Thông tin niêm yết & giao dịch sẽ được truy xuất từ TCBS


-1509.0
Buy by fund (fund_value; -1509.0) executed at 2019-02-26 00:00:00 with 100.0 shares at price 16.84, total price 1684.0
-2121.0
Buy by fund (fund_value; -2121.0) executed at 2020-07-21 00:00:00 with 100.0 shares at price 23.87, total price 2387.0
-5490.0
Buy by fund (fund_value; -5490.0) executed at 2022-09-20 00:00:00 with 100.0 shares at price 61.19, total price 6119.0
   Size  EntryBar  ExitBar  EntryPrice  ExitPrice     PnL  ReturnPct  \
0   100       385     1248       23.87      83.42  5955.0   2.494763   
1   100        34     1248       16.84      83.42  6658.0   3.953682   

   EntryTime   ExitTime  Duration  
0 2020-07-21 2023-12-28 1255 days  
1 2019-02-26 2023-12-28 1766 days  
Results for FPT:
Total investment: 4071.0
Current Shares: 200
Current Equity: 16598.0
RoR: 307.71309260623923
--------------------------------------------------


## LÔ CHẴN VÀ THEO THÁNG và VND

In [62]:
import pandas as pd
from backtesting import Backtest, Strategy
import math
from vnstock3 import Vnstock

def calculate_first_mondays(dates):
        if not isinstance(dates, pd.DatetimeIndex):
            dates = pd.DatetimeIndex(dates)
        dates_series = pd.Series(dates, index=dates)
        mondays = dates_series[dates_series.dt.dayofweek == 0]
        first_mondays = mondays.groupby([mondays.dt.year, mondays.dt.month]).first()
        return set(first_mondays)

class DCA(Strategy):
    average_monthly_income_vnd =  1000  # 10.000.000
    investment_percentage = 0.10  # Percentage of income to invest
    fund = 0
    def init(self):
        self.amount_to_invest = self.average_monthly_income_vnd * self.investment_percentage 
        self.first_mondays = calculate_first_mondays(self.data.index)
        # self.day_of_week = self.I(
        #     lambda x: x,
        #     self.data.Close.s.index.dayofweek,
        #     plot=False,
        # )

    def next(self):
        today = self.data.index[-1]
        self.data.Close[-1] = self.data.Close[-1]*0.1
        if today in self.first_mondays:
            # self.buy(size=math.floor(self.amount_to_invest / self.data.Close[-1]))
            share_price = self.data.Close[-1]
            shares_to_buy = self.amount_to_invest // (share_price)# 16450 => 16.45 VNSTOCK => 1.6450
            if shares_to_buy >= 100:
                shares_to_buy = (shares_to_buy // 100) * 100
                self.buy(size=shares_to_buy)
                #print('no fund')
                #print(f"Buy executed at {self.data.index[-1]} with {shares_to_buy} shares at price {share_price}, total price {share_price * shares_to_buy}")
            else:
                self.fund += self.amount_to_invest
                #print(f'{self.fund} before buy')
                shares_to_buy = self.fund // (share_price)
                if shares_to_buy >= 100:
                    shares_to_buy = (shares_to_buy // 100) * 100
                    self.fund -= shares_to_buy * share_price
                    self.buy(size=shares_to_buy)
                    #print(f"Buy by fund executed at {self.data.index[-1]} with {shares_to_buy} shares at price {share_price}, total price {share_price * shares_to_buy}(ĐƠN VỊ LÀ 10000)")
                    #print(f'{self.fund} after buy')
def run_backtest(stock_symbol):
    # Fetch stock data
    stock_data = Vnstock().stock(symbol=stock_symbol).quote.history(start='2019-01-01', end='2024-01-04')
    stock_data = stock_data.rename(columns={"open": "Open", "high": "High", "low": "Low", "close": "Close", "volume": "Volume"})
    stock_data.set_index('time', inplace=True)
    stock_data.index = pd.to_datetime(stock_data.index)
    # Merge USD/VND data
    #stock_data['usd/vnd'] = usd_vnd_data['Close'] / 1000
    #stock_data['Close'] = stock_data['Close'] / stock_data['usd/vnd']

    # Handle NaN values (choose one method)


    # Run the backtest
    bt = Backtest(
        stock_data,
        DCA,
        trade_on_close=True,
    )
    stats = bt.run()
    #bt.plot(filename=f'{stock_symbol}')
    # Calculate investment details
    trades = stats["_trades"]
    price_paid = trades["Size"] * trades["EntryPrice"]
    total_invested = price_paid.sum()

    current_shares = trades["Size"].sum()
    current_equity = current_shares * stock_data.Close.iloc[-1]
    print(trades)
    print(f"Results for {stock_symbol}:")
    print("Total investment:", total_invested)
    print("Current Shares:", current_shares)
    print("Current Equity:", current_equity)
    print("RoR:", ((current_equity - total_invested) / total_invested)*100)
    print("-" * 50)

# Load USD/VND data
# usd_vnd_data = pd.read_csv('D:/datcom lab/BACKTESTING LIBRARY LEARNING/VND=XCommon.csv')
# #usd_vnd_data = pd.read_csv(('D:/datcom lab/2nd paper/VND=X_Common.csv'))
# usd_vnd_data['Date'] = pd.to_datetime(usd_vnd_data['Date'])
# usd_vnd_data.set_index('Date', inplace=True)
# usd_vnd_data.columns = usd_vnd_data.columns.str.strip()
# List of stock symbols
#stock_symbols = ['SSI', 'EIB', 'HPG']  # Add more stock symbols as needed
stock_symbols = ['FPT']
# Run backtest for each stock
for symbol in stock_symbols:
    run_backtest(symbol)


2024-10-09 22:35:32,447 - vnstock3.common.data.data_explorer - WARNING - Thông tin niêm yết & giao dịch sẽ được truy xuất từ TCBS


    Size  EntryBar  ExitBar  EntryPrice  ExitPrice    PnL  ReturnPct  \
0    100      1230     1250       8.083      8.282   19.9   0.024620   
1    100      1060     1250       5.936      8.282  234.6   0.395216   
2    100       936     1250       5.650      8.282  263.2   0.465841   
3    100       833     1250       5.630      8.282  265.2   0.471048   
4    100       707     1250       5.738      8.282  254.4   0.443360   
5    100       584     1250       4.377      8.282  390.5   0.892164   
6    100       502     1250       3.071      8.282  521.1   1.696841   
7    100       438     1250       2.576      8.282  570.6   2.215062   
8    100       374     1250       2.337      8.282  594.5   2.543860   
9    100       329     1250       2.153      8.282  612.9   2.846725   
10   100       287     1250       2.375      8.282  590.7   2.487158   
11   100       228     1250       2.328      8.282  595.4   2.557560   
12   100       188     1250       2.366      8.282  591.6   2.50